In [ ]:
import os

import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

In [ ]:
# refername = "Arabidopsis_thaliana"
# refername = "Erigeron_breviscapus"
# refername = "Glycine_max"
# refername = "Zea_mays"
refername = "sheetall"

In [ ]:
df = pd.read_csv(f"../output/{refername}_ss.csv")

In [ ]:
def getpath(file_path):
    if os.path.exists(file_path):
        return file_path
    else:
        file_path = file_path[:-4]
        return file_path

In [ ]:
dftrue = df[(df["if_right"]) & (df["Binding"] == 1)]

In [ ]:
def sdf2smile(whichname):
    suppl_p = Chem.SDMolSupplier(
        f"../../data/our_data/2407train/SDF_All/{whichname}_P.sdf"
    )
    suppl_s = Chem.SDMolSupplier(
        f"../../data/our_data/2407train/SDF_All/{whichname}_S.sdf"
    )
    smiles_s = "nofind"
    smiles_p = "nofind"
    for mol in suppl_s:
        if mol is not None:
            smiles_s = Chem.MolToSmiles(mol)
    for mol in suppl_p:
        if mol is not None:
            smiles_p = Chem.MolToSmiles(mol)
    return smiles_s, smiles_p

In [ ]:
errorlist = []
for index, value in dftrue["enzyme"].items():
    # try:
    smiles_s, smiles_p = sdf2smile(value)
    dftrue.loc[index, "smile_s"] = smiles_s
    dftrue.loc[index, "smiles_p"] = smiles_p
    dftrue.loc[index, "smiles_r"] = f"{smiles_s}>>{smiles_p}"
# except:
#     errorlist.append(value)

In [ ]:
set(errorlist)

In [ ]:
dftrue[dftrue["smiles_p"] == "nofind"]["enzyme"].unique()

In [ ]:
dftrue[dftrue["smile_s"] == "nofind"]["enzyme"].unique()

In [ ]:
print(dftrue["substrate"].nunique())
print(dftrue["smile_s"].nunique())
print(dftrue["smiles_p"].nunique())
print(dftrue["smiles_r"].nunique())

In [ ]:
# dataframes = {}
# dataframes[f'{refername}_df'] = pd.read_pickle("../../data/our_data/5foldsdata.pkl")
# dfecfp = dataframes[f'{refername}_df'].drop_duplicates(subset=['substrate'])
# dfecfp = dfecfp[dfecfp["substrate"].isin(dftrue["substrate"].unique())]

# for index, value in dftrue['smile_s'].items():
#     mol = Chem.MolFromSmiles(value)
#     ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
#     dftrue.loc[index, "ECFP"] = ecfp.ToBitString()

# dftrue_ecfp = pd.merge(dftrue, dfecfp, on="ECFP", how="right")

In [ ]:
df_unique = df.drop_duplicates(subset="enzyme", keep="first")
unique_seq = pd.Series(df_unique["seq"])
unique_seq.to_csv(
    f"../others/enzypick/{refername}_enzyme.csv", index=False, header=["Sequence"]
)

In [ ]:
dfsubtrue = dftrue.drop_duplicates(subset="substrate", keep="first")

In [ ]:
with open(f"../others/enzypick/{refername}_sp.csv", "w") as fastaout:
    for index, row in dfsubtrue.iterrows():
        smile_s = row["smile_s"]
        smiles_p = row["smiles_p"]
        substrate = row["substrate"]
        enzyme_s = dftrue[dftrue["substrate"] == substrate]["enzyme"]
        enzyme_id = df_unique[df_unique["enzyme"].isin(enzyme_s)].index.tolist()
        enzyme_id = str(enzyme_id).replace(",", "_").replace(" ", "")
        fastaout.write(f"{smile_s},{smiles_p},{substrate},{enzyme_id}\n")

In [ ]:
with open(f"../others/reme/{refername}.fasta", "w") as fastaout:
    for index, row in df_unique.iterrows():
        nowenzyme = row["enzyme"]
        nowseq = row["seq"]
        fastaout.write(f">{nowenzyme}\n{nowseq}\n")

In [ ]:
dfsubtrue[["substrate", "smiles_r"]].to_csv(
    f"../others/reme/{refername}reme.csv", header=False
)

In [ ]:
with open(f"../others/puepp/{refername}_sp.csv", "w") as fastaout:
    for index, row in dfsubtrue.iterrows():
        smile_s = row["smile_s"]
        substrate = row["substrate"]
        enzyme_s = dftrue[dftrue["substrate"] == substrate]["enzyme"]
        enzyme_id = df_unique[df_unique["enzyme"].isin(enzyme_s)].index.tolist()
        enzyme_id = str(enzyme_id).replace(",", "_").replace(" ", "")
        fastaout.write(f"{smile_s},{smiles_p},{substrate},{enzyme_id}\n")